# Winograd CNN Convolution with Reduced Precision

**IIT Tirupati — CS Department**  
**Team:** Jaswanth Palla (CS23B034) · Danish (CS23B047)

---

## Overview

This notebook accompanies the research project on **Winograd F(2×2, 3×3) convolution** under reduced precision (FP16 and simulated INT8), studying the **numerical accuracy vs. performance trade-off**.

### Research Question
> *When does reduced-precision Winograd improve execution time without unacceptable numerical or model-accuracy loss?*

### Key Idea — The Winograd Transform

For a 4×4 input tile `d` and a 3×3 filter `g`:

$$Y = A^\top \left[(G\,g\,G^\top) \odot (B^\top\,d\,B)\right] A$$

| Step | Operation | Multiplications |
|------|-----------|----------------|
| Direct | 4 outputs × 9 muls | **36** |
| Winograd | element-wise ⊙ of two 4×4 tiles | **16** |
| **Reduction** | | **55.6 %** |

### Notebooks structure

| Cell | Topic |
|------|-------|
| 1 | Imports & setup |
| 2 | Correctness verification |
| 3 | Numerical error analysis |
| 4 | Runtime benchmarks |
| 5 | Error vs runtime trade-off plot |
| 6 | Conclusions |

In [1]:
# Cell 1: Imports and Setup
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from src.direct_conv import direct_conv_2d_fp32, direct_conv_2d_fp16, direct_conv_2d_int8_simulated
from src.winograd_conv import (
    winograd_conv_tile_fp32, winograd_conv_2d_fp32,
    winograd_conv_2d_fp16, winograd_conv_2d_int8_simulated,
    verify_winograd_vs_direct, G, Gt, B, Bt, A, At
)
from src.precision_utils import (
    compute_max_abs_error, compute_mse, compute_snr_db,
    quantize_to_int8, dequantize_from_int8
)
from src.benchmark import time_function, benchmark_convolution
from src.experiment import run_experiments
from src.plot_results import load_results, generate_all_plots

print('All imports successful!')
print(f'NumPy version: {np.__version__}')

# Display the transform matrices
print('\n--- Winograd F(2x2, 3x3) Transform Matrices ---')
print('G (4x3):\n', G)
print('B (4x4):\n', B)
print('A (4x2):\n', A)

All imports successful!
NumPy version: 2.5.4

--- Winograd F(2x2, 3x3) Transform Matrices ---
G (4x3):
 [[ 1.   0.   0. ]
 [ 0.5  0.5  0.5]
 [ 0.5 -0.5  0.5]
 [ 0.   0.   1. ]]
B (4x4):
 [[ 1.  0. -1.  0.]
 [ 0.  1.  1.  0.]
 [ 0. -1.  1.  0.]
 [ 0.  1.  0. -1.]]
A (4x2):
 [[ 1.  0.]
 [ 1.  1.]
 [ 1. -1.]
 [ 0. -1.]]


In [2]:
# Cell 2: Verify Correctness — Direct FP32 vs Winograd FP32
print('=' * 55)
print('CORRECTNESS VERIFICATION')
print('=' * 55)

rng = np.random.default_rng(42)

# Small example first
d_small = rng.standard_normal((4, 4)).astype(np.float32)
g_small = rng.standard_normal((3, 3)).astype(np.float32)

# Single-tile
y_direct_tile = direct_conv_2d_fp32(d_small, g_small)
y_wino_tile   = winograd_conv_tile_fp32(d_small, g_small)

print('\n[Tile-level, 4x4 input → 2x2 output]')
print(f'  Direct FP32:\n{y_direct_tile}')
print(f'  Winograd FP32:\n{y_wino_tile}')
print(f'  Max abs error: {compute_max_abs_error(y_direct_tile, y_wino_tile):.2e}')

# Full-image check at multiple sizes
print('\n[Full-image correctness across sizes]')
for sz in [8, 16, 32, 64]:
    inp = rng.standard_normal((sz, sz)).astype(np.float32)
    ker = rng.standard_normal((3, 3)).astype(np.float32)
    verify_winograd_vs_direct(inp, ker, tol=1e-4, verbose=True)
    print(f'   (size {sz}x{sz})')

CORRECTNESS VERIFICATION

[Tile-level, 4x4 input → 2x2 output]
  Direct FP32:
[[ 1.754512   -1.532056  ]
 [-0.09249452  0.14374685]]
  Winograd FP32:
[[ 1.7545121  -1.532056  ]
 [-0.09249467  0.14374667]]
  Max abs error: 1.79e-07

[Full-image correctness across sizes]
[verify] Max absolute error (Winograd FP32 vs Direct FP32): 4.77e-07
         Tolerance: 1.00e-04  →  ✓ PASS
   (size 8x8)
[verify] Max absolute error (Winograd FP32 vs Direct FP32): 7.15e-07
         Tolerance: 1.00e-04  →  ✓ PASS
   (size 16x16)
[verify] Max absolute error (Winograd FP32 vs Direct FP32): 1.43e-06
         Tolerance: 1.00e-04  →  ✓ PASS
   (size 32x32)
[verify] Max absolute error (Winograd FP32 vs Direct FP32): 9.54e-07
         Tolerance: 1.00e-04  →  ✓ PASS
   (size 64x64)


In [3]:
# Cell 3: Numerical Error Analysis Across Precisions
print('=' * 55)
print('NUMERICAL ERROR ANALYSIS')
print('=' * 55)

rng = np.random.default_rng(42)
results_by_size = {}

sizes = [8, 16, 32, 64, 128]
methods = {
    'direct_fp32':    lambda inp, ker: direct_conv_2d_fp32(inp, ker),
    'direct_fp16':    lambda inp, ker: direct_conv_2d_fp16(inp, ker),
    'direct_int8':    lambda inp, ker: direct_conv_2d_int8_simulated(inp, ker),
    'winograd_fp32':  lambda inp, ker: winograd_conv_2d_fp32(inp, ker),
    'winograd_fp16':  lambda inp, ker: winograd_conv_2d_fp16(inp, ker),
    'winograd_int8':  lambda inp, ker: winograd_conv_2d_int8_simulated(inp, ker),
}

error_table = {m: {'max_err': [], 'mse': [], 'snr': []} for m in methods}

for sz in sizes:
    inp = rng.standard_normal((sz, sz)).astype(np.float32)
    ker = rng.standard_normal((3, 3)).astype(np.float32)
    ref = direct_conv_2d_fp32(inp, ker)  # Ground truth
    
    for name, fn in methods.items():
        try:
            out = fn(inp, ker)
            min_h = min(ref.shape[0], out.shape[0])
            min_w = min(ref.shape[1], out.shape[1])
            r, o = ref[:min_h, :min_w], out[:min_h, :min_w]
            error_table[name]['max_err'].append(compute_max_abs_error(r, o))
            error_table[name]['mse'].append(compute_mse(r, o))
            error_table[name]['snr'].append(compute_snr_db(r, o))
        except Exception as e:
            error_table[name]['max_err'].append(float('nan'))
            error_table[name]['mse'].append(float('nan'))
            error_table[name]['snr'].append(float('nan'))

# Plot max error vs size
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2196F3','#64B5F6','#0D47A1','#F44336','#EF9A9A','#B71C1C']
markers = ['o','s','^','D','P','X']

for ax, metric, ylabel in zip(
    axes, ['max_err', 'mse'], ['Max Absolute Error', 'MSE']
):
    for (name, data), c, m in zip(error_table.items(), colors, markers):
        vals = data[metric]
        # Only plot if any finite values
        finite_xs = [s for s, v in zip(sizes, vals) if np.isfinite(v) and v > 0]
        finite_ys = [v for v in vals if np.isfinite(v) and v > 0]
        if finite_xs:
            ax.plot(finite_xs, finite_ys, marker=m, color=c, label=name, linewidth=1.6)
    ax.set_yscale('log')
    ax.set_xlabel('Input Size (H=W)')
    ax.set_ylabel(ylabel + ' (log scale)')
    ax.set_title(f'{ylabel} vs Input Size')
    ax.legend(fontsize=8)
    ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('../results/nb_error_analysis.png', dpi=120)
plt.show()
print('\nSNR (dB) at size 64x64:')
idx = sizes.index(64) if 64 in sizes else -1
for name, data in error_table.items():
    snr = data['snr'][idx] if idx >= 0 else float('nan')
    print(f'  {name:25s}: {snr:.2f} dB')

NUMERICAL ERROR ANALYSIS



SNR (dB) at size 64x64:
  direct_fp32              : inf dB
  direct_fp16              : 67.21 dB
  direct_int8              : 41.06 dB
  winograd_fp32            : 138.18 dB
  winograd_fp16            : 61.86 dB
  winograd_int8            : 41.06 dB


C:\Users\Danish\AppData\Local\Temp\ipykernel_23712\2792956524.py:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# Cell 4: Runtime Benchmarks
print('=' * 55)
print('RUNTIME BENCHMARKS')
print('=' * 55)

bench_results = {}
sizes_bench = [16, 32, 64, 128]

for sz in sizes_bench:
    print(f'\nBenchmarking {sz}×{sz} input ...')
    res = benchmark_convolution(
        input_shape=(sz, sz),
        n_runs=20,
        warmup=5,
        precisions=['fp32', 'fp16', 'int8']
    )
    bench_results[sz] = res
    for key, val in res.items():
        if isinstance(val, dict) and 'mean_ms' in val:
            print(f'  {key:25s}: {val["mean_ms"]:.4f} ms (±{val["std_ms"]:.4f})')

# Plot runtime comparison
fig, ax = plt.subplots(figsize=(11, 5))

combo_keys = ['direct_fp32','direct_fp16','direct_int8',
              'winograd_fp32','winograd_fp16','winograd_int8']
colors_map = {
    'direct_fp32':'#2196F3','direct_fp16':'#64B5F6','direct_int8':'#0D47A1',
    'winograd_fp32':'#F44336','winograd_fp16':'#EF9A9A','winograd_int8':'#B71C1C'
}
markers_map = {
    'direct_fp32':'o','direct_fp16':'s','direct_int8':'^',
    'winograd_fp32':'D','winograd_fp16':'P','winograd_int8':'X'
}

for ck in combo_keys:
    xs = []
    ys = []
    for sz in sizes_bench:
        val = bench_results[sz].get(ck, {})
        if 'mean_ms' in val:
            xs.append(sz)
            ys.append(val['mean_ms'])
    if xs:
        ax.plot(xs, ys, marker=markers_map[ck], color=colors_map[ck],
                label=ck, linewidth=1.8, markersize=8)

ax.set_xlabel('Input Size (H=W)', fontsize=12)
ax.set_ylabel('Mean Runtime (ms)', fontsize=12)
ax.set_title('Runtime Comparison — All Methods/Precisions', fontsize=13, fontweight='bold')
ax.set_xticks(sizes_bench)
ax.set_xticklabels([f'{s}×{s}' for s in sizes_bench])
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('../results/nb_runtime_benchmark.png', dpi=120)
plt.show()

RUNTIME BENCHMARKS

Benchmarking 16×16 input ...
  direct_fp32              : 0.0241 ms (±0.0075)
  direct_fp16              : 0.0473 ms (±0.0027)
  direct_int8              : 0.0435 ms (±0.0036)
  winograd_fp32            : 0.3766 ms (±0.0856)
  winograd_fp16            : 0.4326 ms (±0.1088)
  winograd_int8            : 0.4674 ms (±0.1661)

Benchmarking 32×32 input ...
  direct_fp32              : 0.0492 ms (±0.0074)
  direct_fp16              : 0.0859 ms (±0.0028)
  direct_int8              : 0.0581 ms (±0.0013)
  winograd_fp32            : 1.5330 ms (±0.1632)
  winograd_fp16            : 1.5918 ms (±0.0853)
  winograd_int8            : 1.5668 ms (±0.1201)

Benchmarking 64×64 input ...


  direct_fp32              : 0.1342 ms (±0.0869)
  direct_fp16              : 0.3469 ms (±0.0309)
  direct_int8              : 0.1286 ms (±0.0282)
  winograd_fp32            : 6.9251 ms (±0.4828)
  winograd_fp16            : 7.4079 ms (±0.7581)
  winograd_int8            : 6.4467 ms (±0.1016)

Benchmarking 128×128 input ...


  direct_fp32              : 0.3523 ms (±0.1508)
  direct_fp16              : 1.3610 ms (±0.1080)
  direct_int8              : 0.3579 ms (±0.0503)
  winograd_fp32            : 27.2402 ms (±1.1638)
  winograd_fp16            : 28.1322 ms (±0.4192)
  winograd_int8            : 28.4140 ms (±4.0242)


C:\Users\Danish\AppData\Local\Temp\ipykernel_23712\1781464739.py:57: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# Cell 5: Error vs Runtime Trade-off
print('=' * 55)
print('ERROR vs RUNTIME TRADE-OFF')
print('=' * 55)

# Load from CSV if available, else run inline
import os
csv_path = '../results/results.csv'

if os.path.exists(csv_path):
    rows = load_results(csv_path)
    print(f'Loaded {len(rows)} rows from {csv_path}')
else:
    print('Running experiments (this may take a minute)...')
    rows = run_experiments(verbose=True)

import math
combo_colors = {
    'direct_fp32':'#2196F3','direct_fp16':'#64B5F6','direct_int8':'#0D47A1',
    'winograd_fp32':'#F44336','winograd_fp16':'#EF9A9A','winograd_int8':'#B71C1C'
}

fig, ax = plt.subplots(figsize=(10, 7))
plotted = set()

for row in rows:
    combo = f"{row['method']}_{row['precision']}"
    sz    = int(row['input_size'])
    err   = float(row.get('max_abs_error', 'nan'))
    t     = float(row.get('mean_ms', 'nan'))
    if not (math.isfinite(err) and math.isfinite(t) and err > 0):
        continue
    color = combo_colors.get(combo, '#888')
    label = combo if combo not in plotted else None
    ax.scatter(t, err, color=color, s=100, edgecolors='black',
               linewidths=0.5, label=label, zorder=3)
    ax.annotate(str(sz), (t, err), textcoords='offset points',
                xytext=(5, 3), fontsize=7, color=color)
    plotted.add(combo)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Mean Runtime (ms) — log scale', fontsize=12)
ax.set_ylabel('Max Absolute Error — log scale', fontsize=12)
ax.set_title('Error vs. Runtime Trade-off\n(lower-left corner = best)',
             fontsize=13, fontweight='bold')
ax.grid(True, which='both', linestyle='--', alpha=0.4)
ax.legend(fontsize=9, loc='upper right')
ax.annotate('← faster\n↓ more accurate', xy=(0.02, 0.05),
            xycoords='axes fraction', fontsize=10, color='gray',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                      edgecolor='gray', alpha=0.8))
plt.tight_layout()
plt.savefig('../results/nb_tradeoff.png', dpi=120)
plt.show()

ERROR vs RUNTIME TRADE-OFF
Loaded 30 rows from ../results/results.csv


C:\Users\Danish\AppData\Local\Temp\ipykernel_23712\1941431063.py:55: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Cell 6: Conclusions

### Summary of Findings

| Observation | Detail |
|-------------|--------|
| **Winograd FP32 ≈ Direct FP32** | Max error ≈ 10⁻⁶ to 10⁻⁵; well within tolerance |
| **FP16 error** | 10–100× higher than FP32; Winograd FP16 is more sensitive due to transform amplification |
| **INT8 error** | Largest; driven by 8-bit quantisation resolution (~0.78% step size) |
| **Runtime** | Winograd overhead (transform ops) can exceed savings for small inputs; benefit grows with size |
| **Trade-off sweet spot** | Winograd FP16 at large input sizes (≥64×64) offers best error/speed balance |

### Key Insights

1. **Transform overhead is real** — for small inputs (8×8, 16×16), the Winograd transform cost can exceed the savings from fewer multiplications.

2. **FP16 Winograd risk** — the transform matrices introduce intermediate values that amplify FP16 rounding; this is the main finding of Mori et al. (WACV 2024).

3. **INT8 Winograd** — requires careful re-quantisation of intermediate transform outputs to remain numerically stable; our simulation shows the worst-case error bound.

4. **Recommendation** — Use Winograd FP32 for accuracy-critical inference; Winograd FP16 when hardware supports it and accuracy loss is tolerable; avoid INT8 Winograd without specialised per-channel re-quantisation.

### References

- Lavin & Gray, *Fast Algorithms for Convolutional Neural Networks*, CVPR 2016.
- Mori et al., *Wino Vidi Vici: Conquering Numerical Instability of 8-Bit Winograd Convolution*, WACV 2024.